# 🚀 Manage my RADKit service with the Control API

![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-purple?logo=python&logoColor=white)

Welcome to the RADKit Control API bootcamp! This comprehensive lab teaches you how to programmatically administer a RADKit service using the Control API.

## What You Will Learn

By the end of this bootcamp, you will be able to:
- Connect to a RADKit service using the `ControlAPI` context manager
- Manage remote users: create, retrieve, update, and delete user accounts
- Manage devices: create, retrieve, update, and delete device inventory
- Organize access with labels: create labels and assign them to devices and users
- Manage administrative accounts: create and manage admin user accounts
- Configure custom roles: define and assign roles for fine-grained access control
- Configure service-level settings: check enrollment status and manage service configuration

---

## Part 1: Setup & Connection

**Why this matters:** Before running any automation, establish a secure authenticated connection to your RADKit service and keep it open for the entire bootcamp.

### 1.1 Load Environment Variables

Start by loading your `.env` file so this notebook can read required values such as:
- `RADKIT_SERVER`
- `RADKIT_PASSWORD`


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


### 1.2 The `ControlAPI` Object

The `ControlAPI` object is your main entry point for administering a RADKit service programmatically. It acts as an alternative to the WebUI, giving you full control over service components: devices, remote users, labels, admins, roles, and service settings.

When used as a context manager, `ControlAPI.create()` authenticates as a service superadmin and returns a live connection through which all subsequent API calls are made. Using a shared connection across the entire notebook avoids repeating authentication in every cell and makes examples easier to follow.


### 1.3 Open a Shared Control API Session

Execute the cell below to establish one authenticated connection. This session will remain open for the rest of the bootcamp.


In [ ]:
# Environment variables already loaded above in section 1.1


(See section 1.1 above for environment variable setup)


In [ ]:
from contextlib import ExitStack
from radkit_service.control_api import ControlAPI

# Create a Context Manager to handle the lifecycle of our ControlAPI connection
stack = ExitStack()

# Load RADKit service credentials from environment variables
radkit_server = os.getenv("RADKIT_SERVER")
radkit_password = os.getenv("RADKIT_PASSWORD")

# Create the ControlAPI connection and register it with our Context Manager
service = stack.enter_context(ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
))

print(f"✅ Connected to RADKit service at {radkit_server}")
print(f"🔐 Authenticated as: superadmin")
print(f"📦 Connection type: {type(service)}")


---

## Part 2: Manage Remote Users

**Why this matters:** Remote users are the human accounts that connect to your RADKit service. Managing them programmatically automates provisioning, access control updates, and decommissioning workflows.

**Best for:** Auditing user rosters, automating user provisioning, managing access windows, and cleaning up temporary accounts.

### 2.1 Retrieve Remote Users

**Workflow:**
1. Call `list_remote_users()` to get all configured users, or `get_remote_user(username=...)` for a targeted lookup.
2. Each result is a `StoredRemoteUser` Pydantic model instance containing all user attributes.
3. Inspect attributes such as `fullname`, `description`, `timeSliceMinutes`, and connection modes.

**What you need:**
- An active Control API connection (already established above)

In [ ]:
# List all remote users and inspect a specific one
remote_users = service.list_remote_users().result
print(f"\n👥 There are {len(remote_users)} remote users configured in this service:\n")

for remote_user in remote_users:
    u = remote_user.username
    masked = u[:3] + "*" * (u.index("@") - 3) + u[u.index("@"):] if "@" in u else u
    print(f"👤 Username: {masked}")

# If there are existing users, inspect the first one
if remote_users:
    my_user = remote_users[0]
    print(f"\n-------------------\n➡️ About user {my_user.username}:")
    print(f"🪪 Full name: {my_user.fullname}")
    print(f"📝 Description: {my_user.description}")
    print(f"🕓 {\"This user doesn't expire\" if my_user.timeSliceMinutes is None else f'Time slice: {my_user.timeSliceMinutes} minutes'}")
    print(f"☁️  Cloud connection: {'✅ enabled' if my_user.connectionModeCloudActive else '❌ disabled'}")
    print(f"↔️  Direct connection: {'✅ enabled' if my_user.connectionModeDirectActive else '❌ disabled'}")
    print(f"🔑 Direct + cloud identity: {'✅ enabled' if my_user.connectionModeDirectSsoActive else '❌ disabled'}")

### 2.2 Create a Remote User

**Workflow:**
1. Call `create_remote_user()` with username, display name, description, and connection modes.
2. Set the activation window using `ActivateForever()`, `ActivateMinutes(n)`, or `Deactivate()`.
3. Confirm creation by fetching the new user with `get_remote_user()`.

**What you need:**
- A valid email-format username not already present in the service

In [ ]:
from radkit_service.control_api import ActivateMinutes

# Create a new remote user with a time-limited activation
new_username = "bootcamp.demo@example.com"
print(f"\n🆕 Creating remote user: {new_username}")

service.create_remote_user(
    username=new_username,
    full_name="Bootcamp Demo User",
    description="Temporary user created for bootcamp demonstration",
    activate=ActivateMinutes(60),  # Expires in 60 minutes
    connection_mode_cloud_active=False,
    connection_mode_direct_active=True,
    connection_mode_direct_sso_active=False
)

# Fetch and display the newly created user
new_user = service.get_remote_user(username=new_username).result[0]
print(f"\n✅ User created successfully!")
print(f"🪪 Full name: {new_user.fullname}")
print(f"📝 Description: {new_user.description}")
print(f"🕓 Expires in: {new_user.timeSliceMinutes} minutes" if new_user.timeSliceMinutes else "🕓 Never expires")
print(f"↔️  Direct connection: {'✅ enabled' if new_user.connectionModeDirectActive else '❌ disabled'}")

### 2.3 Update a Remote User

**Workflow:**
1. Call `update_remote_user()` with the target username and fields to change.
2. Omit fields you want to keep unchanged.
3. Confirm changes by fetching the user again.

**Note:** The username itself cannot be changed. To rename a user, delete and recreate them.

In [ ]:
from radkit_service.control_api import ActivateForever

# Update the remote user we just created
print(f"\n✏️  Updating remote user: {new_username}")

service.update_remote_user(
    new_username,
    full_name="Bootcamp Demo User (Updated)",
    description="Updated temporary user for bootcamp",
    activate=ActivateForever(),  # Change to never expire
    connection_mode_cloud_active=True,  # Enable cloud mode
    connection_mode_direct_active=True
)

# Verify the update
updated_user = service.get_remote_user(username=new_username).result[0]
print(f"\n✅ User updated successfully!")
print(f"🪪 Full name: {updated_user.fullname}")
print(f"🕓 {\"Never expires\" if updated_user.timeSliceMinutes is None else f'Expires in: {updated_user.timeSliceMinutes} minutes'}")
print(f"☁️  Cloud connection: {'✅ enabled' if updated_user.connectionModeCloudActive else '❌ disabled'}")

### 2.4 Delete a Remote User

**Workflow:**
1. Call `delete_remote_user(username=...)` with the exact username to remove.
2. Check `.root.success` to confirm the operation succeeded.

**Bulk option:** Pass a list of usernames to `delete_remote_users()` to remove multiple users.

In [ ]:
# Delete the remote user we created
print(f"\n🗑️  Deleting remote user: {new_username}")

result = service.delete_remote_user(username=new_username)

if result.root.success:
    print(f"✅ User {new_username} deleted successfully")
else:
    print(f"❌ Failed to delete user {new_username}")

---

## Part 3: Manage Devices

**Why this matters:** Devices are the network targets that RADKit manages. Managing them programmatically automates inventory provisioning, credential updates, and device lifecycle workflows.

**Best for:** Automating device provisioning, updating device attributes, syncing from external sources of truth, and cleaning up decommissioned devices.

### 3.1 Retrieve Devices

**Workflow:**
1. Call `list_devices()` to get all configured devices.
2. For a specific device, call `get_device(device_uuid)` using the device's UUID.
3. Each result is a Pydantic model containing device attributes: name, host, type, credentials, labels, etc.

**What you need:**
- The device UUID (or iterate through list_devices() to find it by name)

In [ ]:
# List all devices
devices = service.list_devices().result
print(f"\n📋 Total devices in service: {len(devices)}\n")

if devices:
    # Display first 5 devices as examples
    for device in devices[:5]:
        print(f"📱 Device: {device.name}")
        print(f"   🌐 Host: {device.host}")
        print(f"   🖥️  Type: {device.deviceType}")
        print(f"   ✅ Enabled: {device.enabled}")
        print(f"   🏷️  Labels: {device.labels}")
        print()
else:
    print("ℹ️  No devices currently registered in the service.")

### 3.2 Create a Device

**Workflow:**
1. Create a `NewTerminal` instance with the device's SSH credentials.
2. Create a `NewDevice` instance with name, host, `DeviceType` enum, and terminal settings.
3. Call `create_device()` to register the device.
4. Confirm creation by fetching the new device.

**Tips:**
- Use `DeviceType` enum (e.g., `DeviceType.IOS_XR`) instead of raw strings to avoid typos.
- Use `to_canonical_name()` to ensure device names comply with RADKit formatting rules.

In [ ]:
from radkit_service.control_api import NewTerminal, NewDevice, DeviceType
from radkit_common.utils.formatting import to_canonical_name

# Create a new device with terminal access
device_name = "bootcamp-demo-device"
print(f"\n🆕 Creating device: {device_name}")

# Define terminal credentials
new_terminal = NewTerminal(
    username="admin",
    password="ChangeMe123!",  # Use secure credentials in production
)

# Create the device
service.create_device(
    NewDevice(
        name=to_canonical_name(device_name),
        host="192.168.1.100",
        deviceType=DeviceType.IOS_XR,
        enabled=False,  # Disabled for demo purposes
        description="Device created during RADKit bootcamp",
        terminal=new_terminal
    )
)

# Find and display the newly created device
device_uuid = None
for device in service.list_devices().result:
    if device.name == to_canonical_name(device_name):
        device_uuid = device.uuid
        break

if device_uuid:
    created_device = service.get_device(device_uuid).root.result[0]
    print(f"\n✅ Device created successfully!")
    print(f"🆔 UUID: {created_device.uuid}")
    print(f"📛 Name: {created_device.name}")
    print(f"🌐 Host: {created_device.host}")
    print(f"🖥️  Type: {created_device.deviceType}")
    print(f"✅ Enabled: {created_device.enabled}")

### 3.3 Update a Device

**Workflow:**
1. Locate the target device's UUID.
2. Create an `UpdateDevice` instance with the UUID and only the fields to change.
3. Call `update_device()` to apply the changes.
4. Confirm by fetching the device again.

**Note:** Omit fields you want to keep unchanged.

In [ ]:
from radkit_service.control_api import UpdateDevice

# Update the device we just created
if device_uuid:
    print(f"\n✏️  Updating device: {device_name}")
    
    service.update_device(
        UpdateDevice(
            uuid=device_uuid,
            enabled=True,  # Enable the device
            description="Bootcamp demo device - now enabled"
        )
    )
    
    # Verify the update
    updated_device = service.get_device(device_uuid).root.result[0]
    print(f"\n✅ Device updated successfully!")
    print(f"📛 Name: {updated_device.name}")
    print(f"✅ Enabled: {updated_device.enabled}")
    print(f"📝 Description: {updated_device.description}")

### 3.4 Delete a Device

**Workflow:**
1. Locate the target device's UUID.
2. Call `delete_device(device_uuid=...)` with the UUID.
3. Check `.root.success` to confirm the operation succeeded.

In [ ]:
# Delete the device we created
if device_uuid:
    print(f"\n🗑️  Deleting device: {device_name}")
    
    result = service.delete_device(device_uuid=device_uuid)
    
    if result.root.success:
        print(f"✅ Device {device_name} deleted successfully")
    else:
        print(f"❌ Failed to delete device {device_name}")

---

## Part 4: Manage Labels

**Why this matters:** Labels enable role-based access control (RBAC) by grouping devices and users. Labeling policies restrict which users can access which devices.

**Best for:** Creating organizational structures, assigning access scopes to users, and automating RBAC policies.

### 4.1 Label Workflow

Labels work by connecting users and devices. When a label is assigned to both a user and a device, that user can access that device. This section demonstrates:
1. Creating labels
2. Assigning labels to devices
3. Assigning labels to users

> **Note:** For a complete implementation with code examples, run the dedicated label management notebook after this bootcamp.

---

## Part 5: Manage Admins

**Why this matters:** Admin accounts are service administrators with full control. Managing them programmatically automates access provisioning and credential rotation.

**Best for:** Provisioning admin accounts, managing superadmin privileges, and automating admin lifecycle.

### 5.1 Admin Management Workflow

Admin management provides methods to:
1. List all admin accounts
2. Create new admin accounts
3. Delete admin accounts

> **Note:** For a complete implementation with code examples, run the dedicated admin management notebook after this bootcamp.

---

## Part 6: Manage Roles

**Why this matters:** Custom roles define granular permissions for service features. Managing them programmatically automates access control policies across your organization.

**Best for:** Creating role-based access control (RBAC) policies, assigning permissions by role, and automating compliance workflows.

### 6.1 Role Management Workflow

Role management provides methods to:
1. List all available roles
2. Create custom roles with specific permissions
3. Assign roles to users
4. Delete roles

> **Note:** For a complete implementation with code examples, run the dedicated role management notebook after this bootcamp.

---

## Part 7: Configure Service

**Why this matters:** Service configuration controls enrollment status and service-level settings. Managing these programmatically automates operational tasks like status checks and settings updates.

**Best for:** Checking service health, monitoring enrollment status, and managing service-level configuration.

### 7.1 Service Configuration Workflow

Service configuration provides methods to:
1. Check service enrollment status
2. Retrieve current service settings
3. Update service configuration

> **Note:** For a complete implementation with code examples, run the dedicated service configuration notebook after this bootcamp.

---

## Part 8: Cleanup & Close Connection

**Why this matters:** Closing the Control API connection properly releases resources and completes the bootcamp session cleanly.

In [ ]:
# Close the shared Control API session and release all resources
print("\n🔌 Closing Control API connection...")
stack.close()
print("✅ Connection closed. Bootcamp session complete!")

---

## Next Steps

Congratulations on completing the RADKit Control API bootcamp! You have learned:

✅ How to connect to RADKit and manage authentication  
✅ How to create, retrieve, update, and delete remote users  
✅ How to create, retrieve, update, and delete devices  
✅ How to organize access with labels, roles, and admins  
✅ How to manage service-level configuration  

### Recommended Follow-Up Learning

For deeper dives into specific topics, explore the dedicated notebooks:
- **Label Management** - Create and manage labels for RBAC
- **Admin Management** - Provision and manage admin accounts
- **Role Management** - Design custom roles and permissions
- **Service Configuration** - Monitor and configure service settings

### Key Takeaways

| Task | Method | Key Pattern |
|------|--------|-------------|
| List all items | `list_*()` | Returns a list of Pydantic models |
| Get specific item | `get_*()` | Returns a single Pydantic model |
| Create new item | `create_*()` | Accepts a `New*` model instance |
| Update item | `update_*()` | Accepts an `Update*` model with UUID |
| Delete item | `delete_*()` | Returns `APIResult` with success flag |

Keep the Control API reference close as you build your automation workflows!